# Multi-System MCP with Databricks LLM

## What we will build

We will connect **one LLM to two different MCP servers**:

- **GitHub MCP** — access repositories, files and GitHub information
- **Confluence MCP** — search and retrieve enterprise knowledge

The LLM receives tools from **both MCP servers** and independently chooses which tool to use based on the user's question.

**Flow:** User question → LLM → MCP tool selection → GitHub or Confluence → result → LLM → final answer

### The key learning

The application does not decide whether to use GitHub or Confluence.

> **The LLM decides which available MCP tool is appropriate for the question.**

## Final architecture

```text
                         User
                           │
                           │ Natural-language question
                           ▼
                    ┌─────────────┐
                    │     LLM     │
                    │             │
                    │ Choose tool │
                    └──────┬──────┘
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
       GitHub MCP Client        Confluence MCP Client
              │                         │
              ▼                         ▼
        GitHub MCP Server        Confluence MCP Server
              │                         │
              ▼                         ▼
           GitHub                   Confluence
              │                         │
              └────────────┬────────────┘
                           │
                           ▼
                          LLM
                           │
                           ▼
                     Final Answer
```



## 1. Install and import libraries

We need the MCP client, Databricks SDK and LangChain OpenAI-compatible interface.

`nest_asyncio` is not required for this working synchronous MCP flow.


In [0]:
%pip install -U databricks-mcp databricks-sdk langchain-openai
dbutils.library.restartPython()


In [0]:
import json

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from langchain_openai import ChatOpenAI
from langchain_core.messages import ToolMessage

print("✅ Libraries imported successfully")


## 2. Connect to the Databricks LLM

First, verify that the LLM is working independently of MCP.


In [0]:
token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

llm = ChatOpenAI(
    model="databricks-gpt-oss-120b",
    api_key=token,
    base_url="https://dbc-dd80cadd-09e7.cloud.databricks.com/serving-endpoints",
    temperature=0.5
)

response = llm.invoke(
    "What is Model Context Protocol (MCP)? Explain in one sentence."
)

print(response.content)


## 3. Connect to both MCP servers

We now create **two MCP clients**.

### GitHub MCP
Provides tools for interacting with GitHub.

### Confluence MCP
Provides tools for searching and retrieving information from Confluence.

The important point is that these are **two separate MCP servers**, but our application can expose tools from both to the same LLM.


In [0]:
import nest_asyncio
nest_asyncio.apply()

workspace = WorkspaceClient()

# GitHub MCP
github_mcp = DatabricksMCPClient(
    server_url=(
        "https://dbc-dd80cadd-09e7.cloud.databricks.com/ai-gateway/mcp-services/system.ai.github?o=7474657665682914"
    ),
    workspace_client=workspace
)

# Confluence MCP
confluence_mcp = DatabricksMCPClient(
    server_url=(
        "https://dbc-dd80cadd-09e7.cloud.databricks.com/ai-gateway/mcp-services/system.ai.atlassian?o=7474657665682914"
    ),
    workspace_client=workspace
)

github_tools = github_mcp.list_tools()
confluence_tools = confluence_mcp.list_tools()

print("✅ GitHub MCP connected")
print(f"GitHub tools: {len(github_tools)}")

print("\n✅ Confluence MCP connected")
print(f"Confluence tools: {len(confluence_tools)}")


### Inspect the available tools

Each MCP server advertises the capabilities it provides.

```text
GitHub MCP
   ├── search_repositories
   ├── get_file_contents
   └── ...

Confluence MCP
   ├── search
   ├── getConfluencePage
   └── ...
```

The LLM will later receive the tools from **both lists**.


In [0]:
print("GITHUB TOOLS")
for tool in github_tools:
    print("-", tool.name)

print("\nCONFLUENCE TOOLS")
for tool in confluence_tools:
    print("-", tool.name)


## 4. Test both MCP connections directly

Before involving the LLM, verify that each MCP server works independently.

These are simple connectivity tests. In the final flow, the LLM will choose the tools itself.


In [0]:
# GitHub test
github_result = github_mcp.call_tool(
    "search_repositories",
    {"query": "langchain"}
)

print("GITHUB MCP RESULT:")
print(github_result)


In [0]:
# Confluence test
confluence_result = confluence_mcp.call_tool(
    "search",
    {"query": 'type=page AND title~"AI Product Launch"'}
)

print("CONFLUENCE MCP RESULT:")
print(confluence_result)


### What did we prove?

Both independent paths are working:

**Databricks → GitHub MCP → GitHub**

and

**Databricks → Confluence MCP → Confluence**

Now we combine the tools from both servers and give them to one LLM.


## 5. Give tools from BOTH MCP servers to the LLM

The LLM does not need to know which MCP server owns a tool.

We convert the tools from both servers into the LLM's function-tool format and combine them into one list.

> **All available GitHub and Confluence tools are provided to the LLM.**


In [0]:
all_mcp_tools = github_tools + confluence_tools

llm_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.input_schema
        }
    }
    for tool in all_mcp_tools
]

print(f"✅ Total tools provided to the LLM: {len(llm_tools)}")


## 6. Ask questions from different systems

Now the same LLM can answer questions that require different systems.

### Example 1 — GitHub

> Retrieve the README.md file from `JainMradul/mcp-demo-repo`.

The LLM should select a GitHub tool.

### Example 2 — Confluence

> What is the target launch date of the AI Product Launch project?

The LLM should select the appropriate Confluence tool.

### The important point

We don't write:

```text
if question is about GitHub → use GitHub
if question is about Confluence → use Confluence
```

Instead:

**Question → LLM sees all tools → LLM chooses the appropriate tool**


## 7. Try a question

Change only the question below to test different MCP systems.

Try:
- `What is the target launch date of the AI Product Launch project?`
- `Who is the project owner of AI Product Launch?`
- `Retrieve the README.md file from JainMradul/mcp-demo-repo.`

The code does not need to know whether the question belongs to GitHub or Confluence.


In [0]:
question = """
Compare the technologies used in my Sales Insights Dashboard project
with the technologies mentioned in the AI Product Launch project.
"""

messages = [
    {
        "role": "system",
        "content": """
You are a helpful enterprise research assistant.

You have access to tools from multiple systems through MCP,
including GitHub and Confluence.

Choose the most appropriate tool yourself based on the user's question.
Use the available tools when the answer requires information from GitHub
or Confluence.

If the requested information requires multiple tools, use them sequentially.
Use only information returned by the tools when answering questions about
these enterprise systems.
"""
    },
    {
        "role": "user",
        "content": question
    }
]


## 8. LLM ↔ MCP tool-calling loop

The LLM can select a tool from **either MCP server**.

When a tool is selected, we route the call to the MCP client that owns that tool.

The LLM does not need to know how the routing is implemented.


In [0]:
llm_with_tools = llm.bind_tools(llm_tools)

# Map each tool name to the MCP client that provides it
tool_to_client = {
    tool.name: github_mcp for tool in github_tools
}
tool_to_client.update({
    tool.name: confluence_mcp for tool in confluence_tools
})

MAX_ITERS = 3

for _ in range(MAX_ITERS):
    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        print("\n================================")
        print("FINAL ANSWER")
        print("================================\n")
        print(response.content)
        break

    messages.append(response)

    for call in response.tool_calls:
        print("\nLLM → MCP:", call["name"])
        print(json.dumps(call["args"], indent=2))

        mcp_client = tool_to_client[call["name"]]

        result = mcp_client.call_tool(
            call["name"],
            call["args"]
        )

        print("MCP → LLM: result received")

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )
else:
    print("Stopped: maximum iterations reached.")



### Key takeaway

**MCP allows one AI application to connect an LLM to multiple external systems through standardized tools.**

The application exposes the available tools to the LLM. The LLM determines **which capability it needs** based on the user's natural-language question.

**One LLM + Multiple MCP Servers → Intelligent Tool Selection**
